##SETUP DATA

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/MCU_Case_Analysis/01_Raw_Data/mcu.csv')

Mounted at /content/drive


In [2]:
df.head(10)

,movie_title,mcu_phase,release_date,tomato_meter,audience_score,movie_duration,production_budget,opening_weekend,domestic_box_office,worldwide_box_office
0,Iron Man,Phase 1,2008-05-02,94,91,126.0,140000000.0,102118668.0,319034126.0,5.857962e+08
1,The Incredible Hulk,Phase 1,2008-06-13,67,70,112.0,137500000.0,55414050.0,134806913.0,2.655739e+08
2,Iron Man 2,Phase 1,2010-05-07,72,71,124.0,200000000.0,128122480.0,312433331.0,6.211564e+08
3,Thor,Phase 1,2011-05-06,77,76,114.0,150000000.0,65723338.0,181030624.0,4.493266e+08
4,Captain America The First Avenger,Phase 1,2011-07-22,80,75,124.0,140000000.0,65058524.0,176654505.0,3.705698e+08
5,The Avengers,Phase 1,2012-05-04,91,91,143.0,220000000.0,207438708.0,623357910.0,1.518813e+09
6,Iron Man 3,Phase 2,2013-05-03,79,78,130.0,200000000.0,174144585.0,409013994.0,1.214811e+09
7,Thor The Dark World,Phase 2,2013-11-08,66,75,112.0,170000000.0,85737841.0,206362140.0,6.446025e+08
8,Captain America The Winter Soldier,Phase 2,2014-04-04,90,92,136.0,170000000.0,95023721.0,259766572.0,7.144215e+08
9,Guardians of the Galaxy,Phase 2,2014-08-01,91,92,121.0,170000000.0,94307658.0,333176600.0,7.733501e+08


In [3]:
print(df.shape)
df.info()

(62, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movie_title           62 non-null     object 
 1   mcu_phase             62 non-null     object 
 2   release_date          62 non-null     object 
 3   tomato_meter          62 non-null     int64  
 4   audience_score        62 non-null     int64  
 5   movie_duration        38 non-null     float64
 6   production_budget     38 non-null     float64
 7   opening_weekend       38 non-null     float64
 8   domestic_box_office   38 non-null     float64
 9   worldwide_box_office  38 non-null     float64
dtypes: float64(5), int64(2), object(3)
memory usage: 5.0+ KB


Columns #6-9 have a discrepancy of 24 entries from the previous columns. This is due to the exclusion of Disney+ streaming series

---



In [4]:
df.isna().sum()

,0
movie_title,0
mcu_phase,0
release_date,0
tomato_meter,0
audience_score,0
movie_duration,24
production_budget,24
opening_weekend,24
domestic_box_office,24
worldwide_box_office,24


Title, phase, release date, and both score columns have 0 missing values, as expected since every project has these. The five financial/runtime columns each have exactly 24 missing values, matching the 24 Disney+ series in the dataset. Box office and budget don’t apply to streaming releases, so these gaps are expected and no rows need to be removed.


In [5]:
# formatting numbers to be more accessible.
pd.set_option("display.float_format", "{:,.2f}".format)

In [6]:
df.describe()

,tomato_meter,audience_score,movie_duration,production_budget,opening_weekend,domestic_box_office,worldwide_box_office
count,62.00,62.00,38.00,38.00,38.00,38.00,38.00
mean,82.63,81.21,130.87,"203,460,526.32","137,733,534.84","367,125,087.71","913,754,990.00"
std,12.72,14.04,15.52,"55,599,239.44","77,280,890.21","205,421,836.81","582,387,060.75"
min,46.00,32.00,105.00,"130,000,000.00","46,110,859.00","84,500,223.00","206,136,825.00"
25%,77.00,76.25,118.75,"170,000,000.00","81,564,311.75","208,397,832.25","486,881,376.25"
50%,86.00,86.00,128.50,"200,000,000.00","117,336,165.50","326,105,363.00","767,139,114.00"
75%,91.75,91.00,136.00,"223,750,000.00","180,789,606.25","422,955,281.00","1,147,955,776.00"
max,99.00,98.00,181.00,"365,000,000.00","360,091,572.00","858,400,177.00","2,799,439,100.00"


\No unusual discrepancies found such as an above 100 score on tomato_meter or a negative movie_duration


###Notable Limitations / Concerns
- Sony Distribution: The four Spiderman films are distributed by Sony, which keep most of their box office. Should be noted that results don't fully reflect Disney's revenue.

- Theatrical Revenue Only: The data doesn't include streaming, home video or merchandise revenue, which could tell a different story compared to only theatrical releases.

- Review Bombing: Some projects were headlined as being "review bombed" (such as *She-Hulk* and *The Eternals*), so critic scores should be treated as more stable.
    - sources: Business Insider, MOVIEWEB, The Verge

- COVID Impact: 2021 releases were affected by theater restrictions, and *Black Widow* released on Disney+ the same day as theaters.

- Score Timing: Review scores for recent releases may still change slightly over time.



---


##PROCESSING DATA

---



In [7]:
# determines how many projects were released per MCU phase
df['mcu_phase'].value_counts().sort_index()

,count
mcu_phase,
Phase 1,6
Phase 2,6
Phase 3,11
Phase 4,15
Phase 5,17
Phase 6,7


In [8]:
df["release_date"] = pd.to_datetime(df["release_date"])
df = df.sort_values("release_date")
df = df.reset_index(drop=True)
# added line to reset index as without it, values keep their original index
# numbers depsite ordered by date, next two cells below are created to
# showcase this

df["year"] = df["release_date"].dt.year
# created 'year' column as it will be necessary to group projects by year
df[["movie_title", "release_date", "year"]].head()

,movie_title,release_date,year
0,Iron Man,2008-05-02,2008
1,The Incredible Hulk,2008-06-13,2008
2,Iron Man 2,2010-05-07,2010
3,Thor,2011-05-06,2011
4,Captain America The First Avenger,2011-07-22,2011




---



In [9]:
##EXAMPLE CELL 1
# compare index numbers on left, all out of order here
df.sort_values('movie_title').head()

,movie_title,mcu_phase,release_date,tomato_meter,audience_score,movie_duration,production_budget,opening_weekend,domestic_box_office,worldwide_box_office,year
47,Agatha All Along,Phase 5,2024-09-18,83,81,NaN,NaN,NaN,NaN,NaN,2024
11,Ant-Man,Phase 2,2015-07-17,83,85,117.00,"130,000,000.00","57,225,526.00","180,202,163.00","519,311,965.00",2015
19,Ant-Man and the Wasp,Phase 3,2018-07-06,87,80,118.00,"162,000,000.00","75,812,205.00","216,648,740.00","622,674,139.00",2018
38,Ant-Man and the Wasp Quantumania,Phase 5,2023-02-17,46,82,125.00,"200,000,000.00","106,109,650.00","214,504,909.00","476,071,180.00",2023
10,Avengers Age of Ultron,Phase 2,2015-05-01,76,82,141.00,"365,000,000.00","191,271,109.00","459,005,868.00","1,395,316,979.00",2015


In [10]:
##EXAMPLE CELL 2
# after inserting reset_index func, index numbers are reset to be chronological
df.sort_values('movie_title').reset_index(drop=True).head()

,movie_title,mcu_phase,release_date,tomato_meter,audience_score,movie_duration,production_budget,opening_weekend,domestic_box_office,worldwide_box_office,year
0,Agatha All Along,Phase 5,2024-09-18,83,81,NaN,NaN,NaN,NaN,NaN,2024
1,Ant-Man,Phase 2,2015-07-17,83,85,117.00,"130,000,000.00","57,225,526.00","180,202,163.00","519,311,965.00",2015
2,Ant-Man and the Wasp,Phase 3,2018-07-06,87,80,118.00,"162,000,000.00","75,812,205.00","216,648,740.00","622,674,139.00",2018
3,Ant-Man and the Wasp Quantumania,Phase 5,2023-02-17,46,82,125.00,"200,000,000.00","106,109,650.00","214,504,909.00","476,071,180.00",2023
4,Avengers Age of Ultron,Phase 2,2015-05-01,76,82,141.00,"365,000,000.00","191,271,109.00","459,005,868.00","1,395,316,979.00",2015




---



In [11]:
df["type"] = df["production_budget"].notna().map({True: "film", False: "series"})
df["type"].value_counts()

,count
type,
film,38
series,24


In [12]:
df['distributor'] = 'Disney'
spidey_films = ['Spider-Man Homecoming', 'Spider-Man Far From Home',
                'Spider-Man No Way Home', 'Spider-Man Brand New Day']
df.loc[df['movie_title'].isin(spidey_films), 'distributor'] = 'Sony'
df[df['movie_title'].isin(spidey_films)][['movie_title', 'year']]

,movie_title,year
15,Spider-Man Homecoming,2017
22,Spider-Man Far From Home,2019
31,Spider-Man No Way Home,2021
59,Spider-Man Brand New Day,2026


Added a distributor column for flagging the Spider-Man films as these are distributed by Sony. *These entries will remain in analysis as the films are part of the MCU but the revenue claim belongs mainly to Sony.*

In [13]:
df.loc[df["movie_title"].str.contains("Brand New Day"), "worldwide_box_office"] = 2523147515
df["recovery_ratio"] = (df["worldwide_box_office"] / df["production_budget"]).round(2)

df.loc[df["movie_title"].str.contains("Brand New Day"), "domestic_box_office"] = 959006311

Dataset from Kaggle had listed Spider-Man Brand New Day's domestic and worldwide revenue only up to September. Went ahead and adjusted to reflect most current revenue stas as reported from BoxOfficeMojo.com as of November 13th, 2026.

In [14]:
##SCORE DISCREPANCIES THAT NEED TO BE ADJUSTED

## MOVIE RELEASES
# CAPTAIN AMERICA BRAVE NEW WORLD TOMATO_METER: 46 / AUDIENCE_SCORE: 75
# SPIDER-MAN BRAND NEW DAY: TOMATO_METER: 90 (NOT 89) / AUDIENCE_SCORE: 97 (NOT 93)

## DISNEY+ SERIES
# MARVEL ZOMBIES - AUDIENCE_SCORE: 67 (NOT 75)
# WONDER MAN - AUDIENCE_SCORE: 87 (NOT 82)
# DAREDEVIL BORN AGAIN SEASON 1 - TOMATO_METER: 87 (NOT 86) / AUDIENCE_SCORE: 79 (NOT 88)
# DAREDEVIL BORN AGAIN SEASON 2 - TOMATO_METER: 87 (NOT 90) / AUDIENCE_SCORE: 85 (NOT 89)
# THE PUNISHER ONE LAST KILL - TOMATO_METER: 73 (NOT 87) / AUDIENCE_SCORE: 78 (NOT 90)
# XMEN '87 SEASON 2 - TOMATO_METER: 98 (NOT 96) / AUDIENCE_SCORE: 76 (NOT 92)


In [15]:
# SCORE UPDATES - Rotten Tomatoes, as of Nov 13, 2026

score_updates = {
    "Captain America Brave New World": {"tomato_meter": 46, "audience_score": 75},
    "Spider-Man Brand New Day": {"tomato_meter": 90, "audience_score": 97},
    "Marvel Zombies": {"tomato_meter": 66, "audience_score": 67},
    "Wonder Man": {"audience_score": 87},
    "Daredevil Born Again": {"tomato_meter": 87, "audience_score": 79},
    "Daredevil Born Again Season 2": {"tomato_meter": 87, "audience_score": 85},
    "The Punisher One Last Kill": {"tomato_meter": 73, "audience_score": 78},
    "X-Men '97 Season 2": {"tomato_meter": 98, "audience_score": 76}
}

for title, scores in score_updates.items():
    row = df["movie_title"] == title
    for column, value in scores.items():
        df.loc[row, column] = value

df[df["movie_title"].isin(score_updates.keys())][["movie_title", "tomato_meter", "audience_score"]]

,movie_title,tomato_meter,audience_score
49,Captain America Brave New World,46,75
50,Daredevil Born Again,87,79
55,Marvel Zombies,66,67
56,Wonder Man,91,87
57,Daredevil Born Again Season 2,87,85
58,The Punisher One Last Kill,73,78
59,Spider-Man Brand New Day,90,97
60,X-Men '97 Season 2,98,76


Critic (tomato_meter) and Audience scores adjusted

In [16]:
df["recovery_ratio"] = (df["worldwide_box_office"] / df["production_budget"]).round(2)
films = df[df["type"] == "film"]
films[["movie_title", "year", "recovery_ratio"]].sort_values("recovery_ratio").reset_index(drop=True)

,movie_title,year,recovery_ratio
0,The Marvels,2023,0.76
1,Eternals,2021,1.70
2,Black Widow,2021,1.90
3,The Incredible Hulk,2008,1.93
4,Thunderbolts,2025,2.12
5,Captain America Brave New World,2025,2.31
6,Ant-Man and the Wasp Quantumania,2023,2.38
7,The Fantastic Four First Steps,2025,2.61
8,Captain America The First Avenger,2011,2.65
9,Shang-Chi and the Legend of the Ten Rings,2021,2.88


Added a recovery_ratio (worldwide gross / production budget). A ratio below 2.5 suggests a film likely lost money once marketing and theater splits are included.

Reasoning: The industry rule of thumb is that a film must gross roughly 2.5× its production budget to break even theatrically as studios keep only about half of ticket sales and also must account for marketing costs (can vary wildly but in this case, MCU films are big-budget blockbusters with large consistent marketing campaigns). Variety's reported *Thunderbolts* break-even point was \$425M (2.4x its \$180M budget), which supports using 2.5× as the threshold for all MCU films.

Sources: Variety, IMDbPro, DFIrentals, Geeks+Gamers

In [17]:
df.to_csv("/content/drive/MyDrive/MCU_Case_Analysis/02_Clean_Data/mcu_clean.csv", index = False)
print("Saved:", df.shape)

Saved: (62, 14)


###Processing Log

1.   Converted release_date from text to date format and sorted all projects chronologically
2.   Added year column for grouping by release year
3. Added type column (38 films, 24 series) based on whether a production budget exists
4. Added distributor column to flag the
 4 Sony-distributed Spider-Man films
5. Updated *Spider-Man: Brand New Day* domestic and worldwide gross numbers to reflect most recent revenue from BoxOfficeMojo.com as of Nov 13, 2026
      - Domestic: \$858M -> \$959M
      - Worldwide: \$2.22B -> \$2.52B
6. Updated tomato_meter and audience scores for certain projects to reflect most up-to-date numbers as reported by RottenTomatoes.com
    - Notable Changes
      - X-Men '97 S2 Audience Score [92 -> 76]
      - Punisher One Last Kill Critic Score [87 -> 73]
7. Added recovery_ratio (worldwide gross / production budget).

